# Генерация таблицы данных для M1 пайлайна

In [1]:
import sys
from dotenv import load_dotenv

sys.path.append('./../')
load_dotenv()

True

In [2]:
from typing import Any, List, TypedDict
import json
from langchain_openai import OpenAIEmbeddings
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import numpy as np

from generators.m1_generator import MDGenerationFlow, MDGenerationState, MetadataStructure
#from measurment.judges.semantic_clustering.cluster_namer import ClusterNamerWF, ClusterDescState, ClusterStructure
#from measurment.judges.semantic_clustering.clusters_mapper import ClusterMapperWF, MappingResult, MappingState
#from measurment.judges.semantic_clustering.cluster_comparer import ClusterNamerWF, ClusterDescState, ClusterStructure

/home/alex/Documents/GitHub/mas_workflow/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL = 'gpt-5.4-mini'
MODEL_PROVIDER = 'openai'
TEMPERATURE = 0.5

def read_prompts(pr_path:str) -> str:
    with open(pr_path, 'rt') as fio:
        res = fio.read()
    return res 

In [4]:
import time
from dataclasses import dataclass
from typing import Optional
from langchain_core.callbacks import BaseCallbackHandler


class TokenCounter(BaseCallbackHandler):
    def __init__(self):
        self.prompt_tokens     = 0
        self.completion_tokens = 0
        self.total_tokens      = 0
        self.n_calls           = 0

    def on_llm_end(self, response, **kwargs):
        usage = None

        # LangChain wraps different providers differently
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage") or \
                    response.llm_output.get("usage")

        # OpenAI via langchain_openai exposes it here
        if usage is None and response.generations:
            for gen_list in response.generations:
                for gen in gen_list:
                    info = getattr(gen, "generation_info", None) or {}
                    usage = info.get("usage") or info.get("token_usage")
                    if usage:
                        break

        if usage:
            self.prompt_tokens     += usage.get("prompt_tokens",     0)
            self.completion_tokens += usage.get("completion_tokens", 0)
            self.total_tokens      += usage.get("total_tokens",      0)
        self.n_calls += 1

In [5]:
class QueryDescr(TypedDict):
    id: str
    topic: str
    field_of_science: str
    llm_query: str

def read_tasks(tasks_file: str) -> List[str]:
    with open(tasks_file, 'rt') as fio:
        d = json.load(fio)
        queries_list = []
        for el in d:
            queries_list.append(QueryDescr(el))
    return queries_list

def generate_meta(md_gen_wf: MDGenerationFlow, query: QueryDescr) -> MetadataStructure:
    counter = TokenCounter()

    md_gen_state = MDGenerationState()
    wf = md_gen_wf.workflow
    
    t0 = time.perf_counter()
    md_gen_state['users_task'] = query['llm_query']
    res_state = wf.invoke(
        md_gen_state,
        config={"callbacks": [counter]},
    )
    elapsed = time.perf_counter() - t0

    tokens_dict = {
        'prompt': counter.prompt_tokens,
        'completion': counter.completion_tokens,
        'total': counter.total_tokens
    }

    return res_state, elapsed, tokens_dict


In [6]:
qs = read_tasks("./../queries/exp_tests.yaml")
qs = [qs[-1]]

In [7]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment


def save_metadata_to_xlsx(data: dict[str, dict[str, str]], output_path: str) -> None:
    wb = Workbook()
    ws = wb.active
    ws.title = "Metadata"

    ws.append(["Field Name", "Description", "Type"])
    for cell in ws[1]:
        cell.font = Font(bold=True, name="Arial")

    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 45
    ws.column_dimensions["C"].width = 15

    for field_name, meta in data.items():
        ws.append([field_name, meta.get("descr", ""), meta.get("type", "")])
        ws.cell(ws.max_row, 2).alignment = Alignment(wrap_text=True, vertical="top")

    wb.save(output_path)

In [8]:
import os
save_dir = './exp_results'

md_maker = MDGenerationFlow(read_prompts('./../prompts/md_generation/m1_baseline_prompt.txt'), MODEL, MODEL_PROVIDER, TEMPERATURE)
for i, q in enumerate(qs):
    full_path  = os.path.join(save_dir, q['topic'])
    os.makedirs(full_path, exist_ok=True)
    
    succes = False
    while succes != True:
        try:
            md, el_t, tokens = generate_meta(md_maker, q)
            if len(md['generation']['md_schema']) != 0:
                succes = True
        except Exception as e:
            print(str(e))

    print(i, len(md['generation']['md_schema']), el_t, tokens)
    save_metadata_to_xlsx(md['generation']['md_schema'], os.path.join(full_path, "m1_metadata.xlsx"))


0 36 11.240156420000062 {'prompt': 821, 'completion': 1379, 'total': 2200}


## 1. Generation a lot of metadatas (for averaging results) and thier embeddings

In [ ]:
import pandas as pd
import os
import shutil
import time


QUERIES_BANKS = {
    "synth_bio":'./../queries/bio_queries.yaml',
}

MD_GEN_PROMPTS = {
    'basic': './../prompts/md_generation/m1_baseline_prompt.txt',
}
RUNS_CNT = 8
DEST_DICT = './../data/openai-oss/md_generations/m1'

bank_dicts = []
os.makedirs(DEST_DICT, exist_ok=True)


In [ ]:
# iterate over q_banks...
for q_name, q_path in QUERIES_BANKS.items():
    q_bank = read_tasks(q_path)
    # ...now over queries...
    for q_num, q in enumerate(q_bank):
        # ...also iterate over prompts of generation...
        for pr_name, pr_path in MD_GEN_PROMPTS.items():
            md_maker = MDGenerationFlow(read_prompts(pr_path), MODEL, MODEL_PROVIDER, TEMPERATURE)

            # ...and now over runs for averaging
            for run in range(RUNS_CNT):
                # create md!
                is_complete = False
                while not is_complete:
                    try:
                        tmp_dicts = []
                        md, el_t, tokens = generate_meta(md_maker, q)
                        md = md['generation']

                        # save md
                        for k, v in md['md_schema'].items():
                            new_dict = {
                                'query_id': q_num,
                                'query_topic': q['topic'],
                                'query_field_of_science': q['field_of_science'],
                                'query_llm_query': q['llm_query'],
                                'run_id': run,
                                'md_short_name': md['short_name'],
                                'md_description': md['description'],
                                'field_name': k,
                                'field_description': v['descr'],
                                'field_type': v['type'],
                                'elapsed_time': el_t,
                                'tokens': str(tokens)
                            }
                            tmp_dicts.append(new_dict)

                        if len(tmp_dicts) == 0:
                            print(f'Empty dicts - rework (wasted: {el_t, tokens}).')
                        else:
                            is_complete = True
                    except Exception as e:
                        print(f'Exception {str(e)} catching. Retrying...')
                        time.sleep(120)
                        continue
                bank_dicts += tmp_dicts

                print(f'Finish processing: {os.path.join(q_name, str(q_num), pr_name, f"r_{run+1}")}', 'elapsed_time', el_t, 'tokens', str(tokens))

    df = pd.DataFrame(bank_dicts)
    df.to_csv(os.path.join(DEST_DICT, f'm1_{q_name}_mds.csv'))


## Compute embeds

In [ ]:
import os
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from eval.md_classes import *

QUERIES_CNT = 16
RUNS = 8

df = pd.read_csv('./data/qwen/md_generations/m1_synth_bio_mds.csv', index_col=[0])
openai_embedder = OpenAIEmbeddings(model="text-embedding-3-large")


In [ ]:
os.makedirs('./embeds/m1/', exist_ok=True)

for idx, row in df.iterrows():
    if 'dag_step' in row.keys():
        block = DataStep(row['dag_step'], row['dag_descr'])
    else:
        block = None
    md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block)
    fields_str = str(md_field)

    emb = openai_embedder.embed_query(fields_str)
    emb = np.array(emb)
    np.save(os.path.join('./embeds/m1/', f'{idx}.npy'), emb)
    print(f"(shape {emb.shape}) - '{md_field}' proceded.")